# LightRAG Hybrid Search 실사용 데모
## 로컬 DB + BM25 키워드 인덱스 설정 → 문서 인서트 → 3가지 모드 쿼리 비교

### 시나리오
1. **로컬 DB + BM25 설정**: NanoVectorDB(기본) + BM25 키워드 인덱스 활성화
2. **LightRAG 선언**: `enable_hybrid_search=True`, `hybrid_search_mode` 설정
3. **문서 인서트**: `ainsert_custom_kg`로 KG 삽입 → 임베딩 + BM25 인덱스 동시 생성
4. **저장된 데이터 확인**: 임베딩(VDB), 그래프(GraphML), BM25 키워드 인덱스
5. **3가지 모드 쿼리**: `vector_only`, `keyword_only`, `hybrid` 비교
6. **결과 분석**: 쿼리 유형별 검색 성능 차이 확인

## 1. 환경 설정 및 LLM 엔드포인트 구성

| 항목 | 설정 |
|---|---|
| **LLM** | `qwen-task-pool` @ `http://222.117.133.162:30010/v1` |
| **Embedding** | 간이 단어 빈도 기반 (64차원) |
| **Vector DB** | NanoVectorDB (로컬 파일 기반) |
| **BM25 인덱스** | `rank_bm25` 인메모리 |
| **Graph DB** | NetworkX + GraphML (로컬 파일) |

In [1]:
import sys, os, shutil, json
import numpy as np

sys.path.insert(0, '..')

from lightrag import LightRAG, QueryParam
from lightrag.utils import EmbeddingFunc, Tokenizer, TokenizerInterface
from lightrag.llm.openai import openai_complete_if_cache
from lightrag.bm25_index import BM25Index, reciprocal_rank_fusion

# ============================================================
# LLM 엔드포인트 (OpenAI 호환 API)
# ============================================================
LLM_BASE_URL = 'http://222.117.133.162:30010/v1'
LLM_MODEL    = 'qwen-task-pool'
LLM_API_KEY  = 'asdf'

async def llm_model_func(prompt, system_prompt=None, history_messages=[], keyword_extraction=False, **kwargs):
    try:
        return await openai_complete_if_cache(
            LLM_MODEL, prompt,
            system_prompt=system_prompt,
            history_messages=history_messages,
            api_key=LLM_API_KEY,
            base_url=LLM_BASE_URL,
            **kwargs,
        )
    except Exception as e:
        if 'low_level_keywords' in prompt or 'high_level_keywords' in prompt:
            return '{\"low_level_keywords\": [\"test\"], \"high_level_keywords\": [\"test\"]}'
        return f'[LLM 연결 불가: {type(e).__name__}]'

# ============================================================
# 임베딩 함수 (간이 단어 빈도 기반)
# ============================================================
class SimpleTokenizer(TokenizerInterface):
    def encode(self, content): return content.split()
    def decode(self, tokens): return ' '.join(tokens)

VOCAB = {}
EMB_DIM = 64

def _word_to_idx(word):
    w = word.lower().strip()
    if w not in VOCAB: VOCAB[w] = len(VOCAB)
    return VOCAB[w]

async def mock_embedding(texts):
    result = []
    for text in texts:
        vec = np.zeros(EMB_DIM)
        for word in text.split():
            vec[_word_to_idx(word) % EMB_DIM] += 1.0
        norm = np.linalg.norm(vec)
        if norm > 0: vec = vec / norm
        result.append(vec)
    return np.array(result)

print('환경 설정 완료')
print(f'  LLM: {LLM_BASE_URL} (model={LLM_MODEL})')
print(f'  Embedding: 단어빈도 기반 {EMB_DIM}차원')
print(f'  Vector DB: NanoVectorDB (로컬 파일)')
print(f'  BM25: rank_bm25 (인메모리)')

⠋ 🔄 Updating package: openai

⠙ 🔄 Updating package: openai

⠹ 🔄 Updating package: openai

⠸ 🔄 Updating package: openai

⠼ 🔄 Updating package: openai

...

⠙ 🔄 Updating package: openai

⠹ 🔄 Updating package: openai

⠸ 🔄 Updating package: openai

⠼ 🔄 Updating package: openai

환경 설정 완료
  LLM: http://222.117.133.162:30010/v1 (model=qwen-task-pool)
  Embedding: 단어빈도 기반 64차원
  Vector DB: NanoVectorDB (로컬 파일)
  BM25: rank_bm25 (인메모리)


## 2. LightRAG 선언 (로컬 DB + BM25 활성화)

**핵심 설정:**
- `enable_hybrid_search=True` → BM25 인덱스 생성 활성화
- `hybrid_search_mode` → `vector_only` / `keyword_only` / `hybrid` 중 선택
- 모든 데이터는 `working_dir` 로컬 폴더에 저장됨

In [2]:
WORK_DIR = '/tmp/lightrag_hybrid_demo'

if os.path.exists(WORK_DIR):
    shutil.rmtree(WORK_DIR)
os.makedirs(WORK_DIR)

rag = LightRAG(
    working_dir=WORK_DIR,
    llm_model_func=llm_model_func,
    embedding_func=EmbeddingFunc(
        embedding_dim=EMB_DIM,
        max_token_size=4096,
        func=mock_embedding,
    ),
    tokenizer=Tokenizer('simple-tokenizer', SimpleTokenizer()),
    addon_params={
        'enable_hybrid_search': True,      # BM25 인덱스 활성화
        'hybrid_search_mode': 'hybrid',    # 기본 모드: 벡터 + BM25 합산
    },
)

await rag.initialize_storages()

print('LightRAG 인스턴스 생성 완료')
print(f'  working_dir: {WORK_DIR}')
print(f'  enable_hybrid_search: {rag._addon_params["enable_hybrid_search"]}')
print(f'  hybrid_search_mode:   {rag._addon_params["hybrid_search_mode"]}')
print()
print('로컬 DB 파일들:')
for f in sorted(os.listdir(WORK_DIR)):
    size = os.path.getsize(os.path.join(WORK_DIR, f))
    print(f'  {f:50s} ({size:,} bytes)')

LightRAG 인스턴스 생성 완료
  working_dir: /tmp/lightrag_hybrid_demo
  enable_hybrid_search: True
  hybrid_search_mode:   hybrid

로컬 DB 파일들:


## 3. 문서 인서트 (파이프라인 실행)

`ainsert_custom_kg`로 데이터 삽입 시 **다음이 동시에 생성**됨:
1. 청크 임베딩 → `vdb_chunks.json`
2. 엔티티 임베딩 → `vdb_entities.json`
3. 릴레이션 임베딩 → `vdb_relationships.json`
4. 그래프 저장 → `graph_chunk_entity_relation.graphml`
5. **BM25 인덱스 빌드** → 청크/엔티티/릴레이션 각각 키워드 인덱싱

In [3]:
CUSTOM_KG = {
    'chunks': [
        {'content': 'GPT-4o is a multimodal large language model developed by OpenAI. It can process text, images, and audio inputs simultaneously. GPT-4o achieves state-of-the-art performance on multiple benchmarks including MMLU and HumanEval.', 'source_id': 'doc_openai', 'file_path': 'ai_models_survey.txt'},
        {'content': 'LightRAG is a graph-based Retrieval Augmented Generation system developed by HKUDS research group. It constructs knowledge graphs from documents and uses them for more structured retrieval compared to traditional RAG systems.', 'source_id': 'doc_lightrag', 'file_path': 'rag_systems_review.txt'},
        {'content': 'BM25 (Best Matching 25) is a probabilistic ranking function widely used in information retrieval. It scores documents based on term frequency (TF) and inverse document frequency (IDF). BM25 is the default ranking algorithm in Elasticsearch and Apache Lucene.', 'source_id': 'doc_ir', 'file_path': 'information_retrieval.txt'},
        {'content': 'Vector similarity search uses dense embeddings to find semantically similar documents. Common approaches include cosine similarity and approximate nearest neighbor (ANN) algorithms like HNSW and IVF.', 'source_id': 'doc_vector', 'file_path': 'search_methods.txt'},
        {'content': 'Reciprocal Rank Fusion (RRF) is a method for combining search results from multiple retrieval systems. The formula is RRF(d) = sum(1/(k+rank)) where k is typically 60. RRF is used in hybrid search systems that combine keyword and semantic search.', 'source_id': 'doc_rrf', 'file_path': 'fusion_methods.txt'},
        {'content': 'BERT (Bidirectional Encoder Representations from Transformers) was developed by Google AI. It introduced the masked language model pre-training objective and achieved breakthrough results on the GLUE benchmark.', 'source_id': 'doc_bert', 'file_path': 'ai_models_survey.txt'},
        {'content': 'Knowledge graph construction involves entity extraction, relation extraction, and graph building. Named Entity Recognition (NER) and coreference resolution are key components of the entity extraction pipeline.', 'source_id': 'doc_kg', 'file_path': 'kg_construction.txt'},
        {'content': 'The HKUDS research group at the University of Hong Kong focuses on data science and systems research. Their notable projects include LightRAG for graph-based RAG and various recommender system frameworks.', 'source_id': 'doc_hkuds', 'file_path': 'research_groups.txt'},
    ],
    'entities': [
        {'entity_name': 'GPT-4O', 'entity_type': 'AI_MODEL', 'description': 'OpenAI가 개발한 멀티모달 대규모 언어 모델', 'source_id': 'doc_openai', 'file_path': 'ai_models_survey.txt'},
        {'entity_name': 'OPENAI', 'entity_type': 'ORGANIZATION', 'description': 'GPT 시리즈를 개발한 AI 연구 기업', 'source_id': 'doc_openai', 'file_path': 'ai_models_survey.txt'},
        {'entity_name': 'LIGHTRAG', 'entity_type': 'SOFTWARE', 'description': 'HKUDS가 개발한 그래프 기반 RAG 시스템', 'source_id': 'doc_lightrag', 'file_path': 'rag_systems_review.txt'},
        {'entity_name': 'HKUDS', 'entity_type': 'ORGANIZATION', 'description': '홍콩대학교 데이터사이언스 연구 그룹', 'source_id': 'doc_hkuds', 'file_path': 'research_groups.txt'},
        {'entity_name': 'BM25', 'entity_type': 'ALGORITHM', 'description': '정보 검색에 사용되는 확률적 랭킹 함수', 'source_id': 'doc_ir', 'file_path': 'information_retrieval.txt'},
        {'entity_name': 'ELASTICSEARCH', 'entity_type': 'SOFTWARE', 'description': '오픈소스 분산 검색 엔진', 'source_id': 'doc_ir', 'file_path': 'information_retrieval.txt'},
        {'entity_name': 'BERT', 'entity_type': 'AI_MODEL', 'description': 'Google AI가 개발한 양방향 트랜스포머 모델', 'source_id': 'doc_bert', 'file_path': 'ai_models_survey.txt'},
        {'entity_name': 'GOOGLE_AI', 'entity_type': 'ORGANIZATION', 'description': 'Google의 AI 연구 부문', 'source_id': 'doc_bert', 'file_path': 'ai_models_survey.txt'},
        {'entity_name': 'RRF', 'entity_type': 'ALGORITHM', 'description': 'Reciprocal Rank Fusion', 'source_id': 'doc_rrf', 'file_path': 'fusion_methods.txt'},
        {'entity_name': 'VECTOR_SEARCH', 'entity_type': 'TECHNIQUE', 'description': 'Dense embedding 기반 의미 검색', 'source_id': 'doc_vector', 'file_path': 'search_methods.txt'},
        {'entity_name': 'KNOWLEDGE_GRAPH', 'entity_type': 'TECHNIQUE', 'description': '엔티티와 관계를 그래프로 표현', 'source_id': 'doc_kg', 'file_path': 'kg_construction.txt'},
    ],
    'relationships': [
        {'src_id': 'OPENAI', 'tgt_id': 'GPT-4O', 'description': 'OpenAI가 GPT-4o를 개발함', 'keywords': '개발, 생성', 'weight': 2.0, 'source_id': 'doc_openai', 'file_path': 'ai_models_survey.txt'},
        {'src_id': 'HKUDS', 'tgt_id': 'LIGHTRAG', 'description': 'HKUDS가 LightRAG을 개발함', 'keywords': '개발, 연구', 'weight': 2.0, 'source_id': 'doc_lightrag', 'file_path': 'rag_systems_review.txt'},
        {'src_id': 'LIGHTRAG', 'tgt_id': 'KNOWLEDGE_GRAPH', 'description': 'LightRAG은 지식그래프를 활용', 'keywords': '활용, 검색', 'weight': 1.5, 'source_id': 'doc_lightrag', 'file_path': 'rag_systems_review.txt'},
        {'src_id': 'ELASTICSEARCH', 'tgt_id': 'BM25', 'description': 'Elasticsearch가 BM25를 기본 랭킹 알고리즘으로 사용', 'keywords': '사용, 랭킹', 'weight': 1.5, 'source_id': 'doc_ir', 'file_path': 'information_retrieval.txt'},
        {'src_id': 'GOOGLE_AI', 'tgt_id': 'BERT', 'description': 'Google AI가 BERT를 개발함', 'keywords': '개발, NLP', 'weight': 2.0, 'source_id': 'doc_bert', 'file_path': 'ai_models_survey.txt'},
        {'src_id': 'RRF', 'tgt_id': 'VECTOR_SEARCH', 'description': 'RRF가 벡터 검색 결과를 병합', 'keywords': '병합, 하이브리드', 'weight': 1.5, 'source_id': 'doc_rrf', 'file_path': 'fusion_methods.txt'},
        {'src_id': 'RRF', 'tgt_id': 'BM25', 'description': 'RRF가 BM25 결과를 병합', 'keywords': '병합, 하이브리드', 'weight': 1.5, 'source_id': 'doc_rrf', 'file_path': 'fusion_methods.txt'},
        {'src_id': 'GPT-4O', 'tgt_id': 'BERT', 'description': 'GPT-4o와 BERT는 모두 트랜스포머 기반', 'keywords': '트랜스포머, 언어모델', 'weight': 1.0, 'source_id': 'doc_openai', 'file_path': 'ai_models_survey.txt'},
    ],
}

print(f'삽입할 데이터: 청크 {len(CUSTOM_KG["chunks"])}개, 엔티티 {len(CUSTOM_KG["entities"])}개, 관계 {len(CUSTOM_KG["relationships"])}개')

삽입할 데이터: 청크 8개, 엔티티 11개, 관계 8개


In [4]:
# 파이프라인 실행: 임베딩 + 그래프 + BM25 인덱스 동시 생성
await rag.ainsert_custom_kg(CUSTOM_KG)

print('인서트 완료!')
print()

# BM25 인덱스가 인서트 시점에 자동 빌드되었는지 확인
print('=== BM25 인덱스 상태 (인서트 직후) ===')
for name, idx in [('청크', rag._bm25_chunks), ('엔티티', rag._bm25_entities), ('릴레이션', rag._bm25_relations)]:
    built = idx and idx.is_built
    count = len(idx.corpus_ids) if built else 0
    print(f'  {name:8s}: {"빌드 완료" if built else "미빌드":8s} ({count}개 문서)')
print(f'  stale:    {rag._bm25_stale} (다음 쿼리에서 재빌드 필요 여부)')

인서트 완료!

=== BM25 인덱스 상태 (인서트 직후) ===
  청크      : 빌드 완료    (8개 문서)
  엔티티     : 빌드 완료    (11개 문서)
  릴레이션    : 빌드 완료    (8개 문서)
  stale:    False (다음 쿼리에서 재빌드 필요 여부)


## 4. 저장된 데이터 확인

### 4a. 로컬 DB 파일 (VDB + Graph + KV)

In [5]:
print('=== 로컬 DB 파일 목록 ===')
total_size = 0
for f in sorted(os.listdir(WORK_DIR)):
    fp = os.path.join(WORK_DIR, f)
    size = os.path.getsize(fp)
    total_size += size
    label = ''
    if 'vdb_' in f: label = '  [벡터 DB]'
    elif 'graph_' in f: label = '  [그래프]'
    elif 'kv_' in f: label = '  [KV 저장소]'
    print(f'  {f:55s} {size:>8,} bytes{label}')
print(f'  {"":55s} {"-"*8}')
print(f'  {"Total":55s} {total_size:>8,} bytes')

=== 로컬 DB 파일 목록 ===
  graph_chunk_entity_relation.graphml                        7,770 bytes  [그래프]
  kv_store_text_chunks.json                                  4,858 bytes  [KV 저장소]
  vdb_chunks.json                                            6,341 bytes  [벡터 DB]
  vdb_entities.json                                          7,221 bytes  [벡터 DB]
  vdb_relationships.json                                     5,624 bytes  [벡터 DB]
                                                          --------
  Total                                                     31,814 bytes


### 4b. 그래프 (GraphML) 확인

In [6]:
import networkx as nx

graph_path = os.path.join(WORK_DIR, 'graph_chunk_entity_relation.graphml')
G = nx.read_graphml(graph_path)

print(f'=== 지식 그래프 ===')
print(f'  노드(엔티티): {G.number_of_nodes()}개')
print(f'  엣지(관계):  {G.number_of_edges()}개')
print()
print('엔티티 목록:')
for node in sorted(G.nodes()):
    data = G.nodes[node]
    print(f'  - {node:20s} (type={data.get("entity_type", "?")})')
print()
print('관계 목록:')
for src, tgt in G.edges():
    data = G.edges[src, tgt]
    print(f'  {src:20s} -> {tgt:20s} | {data.get("description", "")[:40]}')

=== 지식 그래프 ===
  노드(엔티티): 11개
  엣지(관계):  8개

엔티티 목록:
  - BERT                 (type=AI_MODEL)
  - BM25                 (type=ALGORITHM)
  - ELASTICSEARCH        (type=SOFTWARE)
  - GOOGLE_AI            (type=ORGANIZATION)
  - GPT-4O               (type=AI_MODEL)
  - HKUDS                (type=ORGANIZATION)
  - KNOWLEDGE_GRAPH      (type=TECHNIQUE)
  - LIGHTRAG             (type=SOFTWARE)
  - OPENAI               (type=ORGANIZATION)
  - RRF                  (type=ALGORITHM)
  - VECTOR_SEARCH        (type=TECHNIQUE)

관계 목록:
  GPT-4O               -> OPENAI               | OpenAI가 GPT-4o를 개발함
  GPT-4O               -> BERT                 | GPT-4o와 BERT는 모두 트랜스포머 기반
  LIGHTRAG             -> HKUDS                | HKUDS가 LightRAG을 개발함
  LIGHTRAG             -> KNOWLEDGE_GRAPH      | LightRAG은 지식그래프를 활용
  BM25                 -> ELASTICSEARCH        | Elasticsearch가 BM25를 기본 랭킹 알고리즘으로 사용
  BM25                 -> RRF                  | RRF가 BM25 결과를 병합
  BERT                 -> GOOGLE_AI  

### 4c. 벡터 DB (임베딩) 확인

In [7]:
for vdb_name in ['vdb_entities.json', 'vdb_relationships.json', 'vdb_chunks.json']:
    vdb_path = os.path.join(WORK_DIR, vdb_name)
    with open(vdb_path, 'r') as f:
        vdb_data = json.load(f)
    records = vdb_data.get('data', [])
    print(f'=== {vdb_name} ({len(records)}개 레코드) ===')
    for rec in records[:3]:
        rec_id = rec.get('entity_name') or rec.get('__id__', '?')
        content = rec.get('content', '')[:60]
        has_vec = '__vector__' in rec
        print(f'  ID: {str(rec_id):25s} | 임베딩: {"있음" if has_vec else "없음"} | {content}...')
    if len(records) > 3:
        print(f'  ... 외 {len(records)-3}개 더')
    print()

=== vdb_entities.json (11개 레코드) ===
  ID: GPT-4O                    | 임베딩: 없음 | GPT-4O
OpenAI가 개발한 멀티모달 대규모 언어 모델...
  ID: OPENAI                    | 임베딩: 없음 | OPENAI
GPT 시리즈를 개발한 AI 연구 기업...
  ID: LIGHTRAG                  | 임베딩: 없음 | LIGHTRAG
HKUDS가 개발한 그래프 기반 RAG 시스템...
  ... 외 8개 더

=== vdb_relationships.json (8개 레코드) ===
  ID: rel-91524ebce3ce96fba5b0557294e81854 | 임베딩: 없음 | 개발, 생성	GPT-4O
OPENAI
OpenAI가 GPT-4o를 개발함...
  ID: rel-6ca47ec01c762b5ec69fa2fc85a3f85b | 임베딩: 없음 | 개발, 연구	HKUDS
LIGHTRAG
HKUDS가 LightRAG을 개발함...
  ID: rel-0f9ddfc3c5b609e00b2401a0a13e4125 | 임베딩: 없음 | 활용, 검색	KNOWLEDGE_GRAPH
LIGHTRAG
LightRAG은 지식그래프를 활용...
  ... 외 5개 더

=== vdb_chunks.json (8개 레코드) ===
  ID: chunk-498ef3524ff4f231b43d502b05853d95 | 임베딩: 없음 | GPT-4o is a multimodal large language model developed by Ope...
  ID: chunk-64ebde6e16b3f9c427e8bc0c6137356d | 임베딩: 없음 | LightRAG is a graph-based Retrieval Augmented Generation sys...
  ID: chunk-f1e105c34c6ff851226f2c278c1eaa31 | 임베딩: 없음 | BM25 (Best Matc

### 4d. BM25 키워드 인덱스 확인 (우리가 추가한 것)

In [8]:
print('=== BM25 키워드 인덱스 상세 ===')
print()

# 청크 BM25
print(f'--- 청크 BM25 ({len(rag._bm25_chunks.corpus_ids)}개) ---')
for cid in rag._bm25_chunks.corpus_ids[:3]:
    text = rag._bm25_chunks.corpus_texts.get(cid, '')[:60]
    print(f'  {cid[:25]:25s} -> "{text}..."')
print()

# 엔티티 BM25
print(f'--- 엔티티 BM25 ({len(rag._bm25_entities.corpus_ids)}개) ---')
for eid in rag._bm25_entities.corpus_ids:
    text = rag._bm25_entities.corpus_texts.get(eid, '')[:50]
    print(f'  {eid:25s} -> "{text}..."')
print()

# 릴레이션 BM25
print(f'--- 릴레이션 BM25 ({len(rag._bm25_relations.corpus_ids)}개) ---')
for rid in rag._bm25_relations.corpus_ids:
    text = rag._bm25_relations.corpus_texts.get(rid, '')[:50]
    print(f'  {rid:30s} -> "{text}..."')

print()
print('직접 BM25 쿼리 테스트:')
for q in ['GPT-4o', 'HKUDS', 'BM25 Elasticsearch']:
    ent_r = rag._bm25_entities.query(q, top_k=2)
    chk_r = rag._bm25_chunks.query(q, top_k=1)
    print(f'  "{q}"')
    print(f'    엔티티: {[r["id"] for r in ent_r]}')
    print(f'    청크:   {[r["id"][:20]+"..." for r in chk_r]}')

=== BM25 키워드 인덱스 상세 ===

--- 청크 BM25 (8개) ---
  chunk-498ef3524ff4f231b43 -> "GPT-4o is a multimodal large language model developed by Ope..."
  chunk-64ebde6e16b3f9c427e -> "LightRAG is a graph-based Retrieval Augmented Generation sys..."
  chunk-f1e105c34c6ff851226 -> "BM25 (Best Matching 25) is a probabilistic ranking function ..."

--- 엔티티 BM25 (11개) ---
  GPT-4O                    -> "GPT-4O
OpenAI가 개발한 멀티모달 대규모 언어 모델..."
  OPENAI                    -> "OPENAI
GPT 시리즈를 개발한 AI 연구 기업..."
  LIGHTRAG                  -> "LIGHTRAG
HKUDS가 개발한 그래프 기반 RAG 시스템..."
  HKUDS                     -> "HKUDS
홍콩대학교 데이터사이언스 연구 그룹..."
  BM25                      -> "BM25
정보 검색에 사용되는 확률적 랭킹 함수..."
  ELASTICSEARCH             -> "ELASTICSEARCH
오픈소스 분산 검색 엔진..."
  BERT                      -> "BERT
Google AI가 개발한 양방향 트랜스포머 모델..."
  GOOGLE_AI                 -> "GOOGLE_AI
Google의 AI 연구 부문..."
  RRF                       -> "RRF
Reciprocal Rank Fusion..."
  VECTOR_SEARCH             -> "VECTOR_SEARCH
Den

## 5. 3가지 모드별 쿼리 비교

| 모드 | 설명 |
|---|---|
| `vector_only` | 기존 벡터 유사도 검색만 사용 |
| `keyword_only` | BM25 키워드 검색만 사용 |
| `hybrid` | 벡터 + BM25 결과를 RRF로 병합 |

`aquery_data`는 LLM 응답 생성 없이 **검색 결과만** 반환하므로 순수한 검색 성능 비교가 가능합니다.

In [9]:
async def compare_3modes(query, mode='naive', description=''):
    """3가지 검색 모드 비교"""
    print('=' * 70)
    print(f'쿼리: "{query}"')
    print(f'모드: {mode} | {description}')
    print('=' * 70)
    
    param = QueryParam(mode=mode, top_k=5)
    results = {}
    
    for search_mode in ['vector_only', 'keyword_only', 'hybrid']:
        rag._addon_params['hybrid_search_mode'] = search_mode
        r = await rag.aquery_data(query, param=param)
        chunks = []
        if r.get('status') == 'success':
            chunks = r.get('data', {}).get('chunks', [])
        results[search_mode] = chunks
    
    for search_mode in ['vector_only', 'keyword_only', 'hybrid']:
        chunks = results[search_mode]
        label = {'vector_only': '벡터 전용', 'keyword_only': '키워드 전용 (BM25)', 'hybrid': '하이브리드 (RRF)'}[search_mode]
        print(f'\n--- [{label}] {len(chunks)}개 청크 ---')
        for c in chunks[:3]:
            content = c.get('content', '')[:70]
            print(f'  - {content}...')
        if not chunks:
            print('  (결과 없음)')
    
    # 차이점 요약
    v_set = {c.get('content','')[:50] for c in results['vector_only']}
    k_set = {c.get('content','')[:50] for c in results['keyword_only']}
    h_set = {c.get('content','')[:50] for c in results['hybrid']}
    
    only_keyword = k_set - v_set
    only_vector = v_set - k_set
    if only_keyword:
        print(f'\n  [BM25만 찾은 청크]: {len(only_keyword)}개')
    if only_vector:
        print(f'  [벡터만 찾은 청크]: {len(only_vector)}개')
    print(f'  [하이브리드 총 청크]: {len(h_set)}개 (벡터+BM25 통합)')
    print()
    
    rag._addon_params['hybrid_search_mode'] = 'hybrid'

In [10]:
await compare_3modes('GPT-4o', mode='naive', description='고유명사 - BM25가 정확한 토큰 매칭으로 강점')

쿼리: "GPT-4o"
모드: naive | 고유명사 - BM25가 정확한 토큰 매칭으로 강점



--- [벡터 전용] 1개 청크 ---
  - GPT-4o is a multimodal large language model developed by OpenAI. It ca...

--- [키워드 전용 (BM25)] 1개 청크 ---
  - GPT-4o is a multimodal large language model developed by OpenAI. It ca...

--- [하이브리드 (RRF)] 1개 청크 ---
  - GPT-4o is a multimodal large language model developed by OpenAI. It ca...
  [하이브리드 총 청크]: 1개 (벡터+BM25 통합)



In [11]:
await compare_3modes('HKUDS LightRAG', mode='naive', description='기관명 + 소프트웨어명 복합 검색')

쿼리: "HKUDS LightRAG"
모드: naive | 기관명 + 소프트웨어명 복합 검색



--- [벡터 전용] 2개 청크 ---
  - LightRAG is a graph-based Retrieval Augmented Generation system develo...
  - The HKUDS research group at the University of Hong Kong focuses on dat...

--- [키워드 전용 (BM25)] 2개 청크 ---
  - The HKUDS research group at the University of Hong Kong focuses on dat...
  - LightRAG is a graph-based Retrieval Augmented Generation system develo...

--- [하이브리드 (RRF)] 2개 청크 ---
  - LightRAG is a graph-based Retrieval Augmented Generation system develo...
  - The HKUDS research group at the University of Hong Kong focuses on dat...
  [하이브리드 총 청크]: 2개 (벡터+BM25 통합)



In [12]:
await compare_3modes('BM25 Elasticsearch ranking', mode='naive', description='기술 용어 검색 - 정확한 키워드 포함 문서 우선')

쿼리: "BM25 Elasticsearch ranking"
모드: naive | 기술 용어 검색 - 정확한 키워드 포함 문서 우선

--- [벡터 전용] 4개 청크 ---
  - BM25 (Best Matching 25) is a probabilistic ranking function widely use...
  - BERT (Bidirectional Encoder Representations from Transformers) was dev...
  - Knowledge graph construction involves entity extraction, relation extr...

--- [키워드 전용 (BM25)] 1개 청크 ---
  - BM25 (Best Matching 25) is a probabilistic ranking function widely use...

--- [하이브리드 (RRF)] 4개 청크 ---
  - BM25 (Best Matching 25) is a probabilistic ranking function widely use...
  - BERT (Bidirectional Encoder Representations from Transformers) was dev...
  - Knowledge graph construction involves entity extraction, relation extr...
  [벡터만 찾은 청크]: 3개
  [하이브리드 총 청크]: 4개 (벡터+BM25 통합)



In [13]:
await compare_3modes(
    'How do AI systems understand and process natural language?',
    mode='naive',
    description='서술형 쿼리 - 벡터 검색이 의미 기반으로 강점'
)

쿼리: "How do AI systems understand and process natural language?"
모드: naive | 서술형 쿼리 - 벡터 검색이 의미 기반으로 강점

--- [벡터 전용] 5개 청크 ---
  - LightRAG is a graph-based Retrieval Augmented Generation system develo...
  - Reciprocal Rank Fusion (RRF) is a method for combining search results ...
  - The HKUDS research group at the University of Hong Kong focuses on dat...

--- [키워드 전용 (BM25)] 8개 청크 ---
  - BERT (Bidirectional Encoder Representations from Transformers) was dev...
  - GPT-4o is a multimodal large language model developed by OpenAI. It ca...
  - The HKUDS research group at the University of Hong Kong focuses on dat...

--- [하이브리드 (RRF)] 8개 청크 ---
  - LightRAG is a graph-based Retrieval Augmented Generation system develo...
  - Reciprocal Rank Fusion (RRF) is a method for combining search results ...
  - The HKUDS research group at the University of Hong Kong focuses on dat...

  [BM25만 찾은 청크]: 3개
  [하이브리드 총 청크]: 8개 (벡터+BM25 통합)



## 6. RRF 점수 계산 시각화

In [14]:
import pandas as pd

vector_results = [
    {'id': 'GPT-4O', 'score': 0.92},
    {'id': 'BERT', 'score': 0.88},
    {'id': 'LIGHTRAG', 'score': 0.81},
    {'id': 'VECTOR_SEARCH', 'score': 0.75},
]
bm25_results = [
    {'id': 'LIGHTRAG', 'score': 8.5},
    {'id': 'HKUDS', 'score': 6.2},
    {'id': 'GPT-4O', 'score': 4.1},
    {'id': 'BM25', 'score': 3.8},
]

k = 60
vec_rank = {r['id']: i for i, r in enumerate(vector_results, 1)}
bm25_rank = {r['id']: i for i, r in enumerate(bm25_results, 1)}
all_ids = set(vec_rank) | set(bm25_rank)

rows = []
for doc in all_ids:
    vr = vec_rank.get(doc); br = bm25_rank.get(doc)
    vs = 1/(k+vr) if vr else 0; bs = 1/(k+br) if br else 0
    rows.append({
        '엔티티': doc,
        'Vector 순위': vr or '-', 'BM25 순위': br or '-',
        'Vector RRF': f'{vs:.6f}' if vr else '-',
        'BM25 RRF': f'{bs:.6f}' if br else '-',
        '합산 RRF': f'{vs+bs:.6f}',
        '출처': 'BOTH' if (vr and br) else ('Vector' if vr else 'BM25'),
    })

df = pd.DataFrame(rows).sort_values('합산 RRF', ascending=False).reset_index(drop=True)
df.index = df.index + 1
df.index.name = '최종 순위'
print('RRF 점수 계산 상세 (k=60)')
print()
print('핵심: BOTH(양쪽 모두 검색된 문서)가 가장 높은 RRF 점수를 받음')
df

RRF 점수 계산 상세 (k=60)

핵심: BOTH(양쪽 모두 검색된 문서)가 가장 높은 RRF 점수를 받음


,엔티티,Vector 순위,BM25 순위,Vector RRF,BM25 RRF,합산 RRF,출처
최종 순위,,,,,,,
1,GPT-4O,1,3,0.016393,0.015873,0.032266,BOTH
2,LIGHTRAG,3,1,0.015873,0.016393,0.032266,BOTH
3,HKUDS,-,2,-,0.016129,0.016129,BM25
4,BERT,2,-,0.016129,-,0.016129,Vector
5,VECTOR_SEARCH,4,-,0.015625,-,0.015625,Vector
6,BM25,-,4,-,0.015625,0.015625,BM25


In [15]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

fused = reciprocal_rank_fusion(vector_results, bm25_results, k=60)
docs_order = [r['id'] for r in fused]
vec_ranks = [vec_rank.get(d, len(vector_results)+2) for d in docs_order]
bm25_ranks = [bm25_rank.get(d, len(bm25_results)+2) for d in docs_order]
fused_ranks = list(range(1, len(docs_order)+1))

fig, ax = plt.subplots(figsize=(12, 6))
x = np.arange(len(docs_order))
width = 0.25
ax.bar(x - width, vec_ranks, width, label='Vector Rank', color='#BBDEFB', edgecolor='#1565C0', linewidth=1.5)
ax.bar(x, bm25_ranks, width, label='BM25 Rank', color='#FFCCBC', edgecolor='#E64A19', linewidth=1.5)
ax.bar(x + width, fused_ranks, width, label='RRF Fused Rank', color='#C8E6C9', edgecolor='#2E7D32', linewidth=1.5)
ax.set_xlabel('Entities', fontsize=12)
ax.set_ylabel('Rank (lower = better)', fontsize=12)
ax.set_title('Vector vs BM25 vs Hybrid(RRF) Ranking Comparison', fontsize=14, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(docs_order, rotation=30, ha='right')
ax.legend(fontsize=10)
ax.invert_yaxis()
ax.set_ylim(max(max(vec_ranks), max(bm25_ranks)) + 0.5, 0.5)
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig('images/ranking_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print('차트 저장: images/ranking_comparison.png')

차트 저장: images/ranking_comparison.png


## 7. 설정 확인 및 정리

In [16]:
from lightrag.addon_params import default_addon_params

defaults = default_addon_params()
print('=== addon_params 기본값 ===')
for k, v in defaults.items():
    if k != 'chunker':
        print(f'  {k}: {v}')

print(f'\nenable_hybrid_search 기본값: {defaults.get("enable_hybrid_search")}')
print(f'hybrid_search_mode 기본값:   {defaults.get("hybrid_search_mode")}')
assert defaults.get('enable_hybrid_search') == False
print('\n기본값이 False/hybrid -> 기존 동작에 영향 없음 확인!')

await rag.finalize_storages()
print('\n스토리지 정리 완료')

=== addon_params 기본값 ===
  language: English
  entity_type_prompt_file: 
  enable_hybrid_search: False
  hybrid_search_mode: hybrid

enable_hybrid_search 기본값: False
hybrid_search_mode 기본값:   hybrid

기본값이 False/hybrid -> 기존 동작에 영향 없음 확인!

스토리지 정리 완료


## 결론

### 3가지 검색 모드 비교

| 모드 | 방식 | 강점 |
|---|---|---|
| `vector_only` | 코사인 유사도 기반 | 의미 유사 문서 검색, 서술형 쿼리 |
| `keyword_only` | BM25 TF-IDF 기반 | 고유명사, 약어, 정확한 키워드 매칭 |
| `hybrid` | 벡터 + BM25 + RRF 병합 | **두 방식의 장점 통합** |

### 파이프라인 요약

```
문서 인서트 → 청크 분할 → 임베딩(VDB) + BM25 인덱스 동시 생성
                  → 엔티티/릴레이션 추출 → 임베딩(VDB) + BM25 인덱스 동시 생성
                  → 그래프 저장 (GraphML)

쿼리 → hybrid_search_mode에 따라:
         vector_only  → VDB 쿼리만
         keyword_only → BM25 쿼리만
         hybrid       → VDB + BM25 → RRF 병합
```

### LLM 엔드포인트

| 항목 | 값 |
|---|---|
| Base URL | `http://222.117.133.162:30010/v1` |
| 모델명 | `qwen-task-pool` |
| API Key | `asdf` |